# LIBRARY IMPORT

In [29]:
from Crypto.PublicKey import RSA # Import the RSA class to generate public and private keys
from Crypto.Cipher import PKCS1_OAEP # Import PKCS1_OAEP for RSA encryption
from Crypto.Cipher import AES # Import AES for symmetric encryption
import os

# CREATING RSA KEYS FOR EACH NETWORK NODE

In [30]:
def generate_RSA_keys():
    RSA_key=RSA.generate(2048) # Generates an RSA key pair with a 2048-bit length
    private_RSA_key=RSA_key.export_key() # Exports the private key
    public_RSA_key=RSA_key.publickey().export_key() # Exports the public key
    return private_RSA_key, public_RSA_key

In [31]:
def indicate_number_network_nodes():
    global number_network_nodes
    number_network_nodes=int(input("Enter the number of network nodes (layers) you want to use (remember, Tor usually only uses 3 nodes in most cases): "))
    print(f"The number of network nodes (layers) chosen is: {number_network_nodes}")
    confirm_message=input(f"Is '{number_network_nodes}' the correct number chosen (answer only with Y/N)? ").upper()
    if confirm_message=="Y":
        print("Answer Y has been selected. Then the program will proceed with the other necessary questions.")
    elif confirm_message=="N":
        print("Answer N has been selected. So, is possible rewrite the number of network nodes (layers) you want to use in the Tor network.")
        indicate_number_network_nodes()
    else:
        print("Invalid response. You can only respond with Y/N. Then the program will ask you to re-enter the number of network nodes (layers) you want to send in the Tor network.")
        indicate_number_network_nodes()
indicate_number_network_nodes()

The number of network nodes (layers) chosen is: 3
Invalid response. You can only respond with Y/N. Then the program will ask you to re-enter the number of network nodes (layers) you want to send in the Tor network.
The number of network nodes (layers) chosen is: 3
Answer N has been selected. So, is possible rewrite the number of network nodes (layers) you want to use in the Tor network.
The number of network nodes (layers) chosen is: 3
Answer Y has been selected. Then the program will proceed with the other necessary questions.


In [32]:
network_nodes=[generate_RSA_keys() for _ in range(number_network_nodes)] # Create 3 network nodes with different keys

In [33]:
public_RSA_key=[network_node[1] for network_node in network_nodes] # Extract only the public keys
private_RSA_key=[network_node[0] for network_node in network_nodes] # Extract only the private keys

# ENCRYPTION WITH AES KEY

In [34]:
def generate_AES_key():
    return os.urandom(16) # Generates a random AES key (128 bits = 16 bytes)

In [35]:
def encrypt_with_AES(data, AES_key):
    cipher=AES.new(AES_key, AES.MODE_EAX)
    cipher_text, cipher_tag=cipher.encrypt_and_digest(data) # Encrypts the data and generates an authentication tag
    return cipher.nonce, cipher_text, cipher_tag

In [36]:
def decrypt_with_AES(nonce, cipher_text, cipher_tag, AES_key):
    cipher=AES.new(AES_key, AES.MODE_EAX, nonce=nonce)
    return cipher.decrypt_and_verify(cipher_text, cipher_tag) # Decrypts the data and verifies the authentication tag

# ENCRYPTION WITH RSA KEY

In [37]:
def encrypt_with_RSA(data, public_RSA_key):
    RSA_key=RSA.import_key(public_RSA_key) # Imports the RSA public key
    cipher=PKCS1_OAEP.new(RSA_key)
    return cipher.encrypt(data) # Encrypts the data with the public key

In [38]:
def decrypt_with_RSA(encrypted_data, private_RSA_key):
    RSA_key=RSA.import_key(private_RSA_key) # Imports the RSA private key
    cipher=PKCS1_OAEP.new(RSA_key)
    return cipher.decrypt(encrypted_data) # Decrypts the data with the private key

# CREATING A MESSAGE WITH ONION ENCRYPTION

In [39]:
def create_onion_message(message, public_keys):
    # Generate an AES key for each node based on the number of public keys
    AES_keys=[generate_AES_key() for _ in public_keys]
    data=message.encode() # Convert the message to bytes
    # Iterate over the public keys in reverse order to apply onion encryption
    for i in reversed(range(len(public_keys))):
        # Encrypt the data using AES with the current key
        nonce, cipher_text, cipher_tag=encrypt_with_AES(data, AES_keys[i])
        # Encrypt the AES key using RSA with the current public key
        encrypted_AES_key=encrypt_with_RSA(AES_keys[i], public_keys[i])
        # Concatenate the encrypted AES key, nonce, ciphertext, and tag to form the onion message
        data=encrypted_AES_key+nonce+cipher_text+cipher_tag
    return data # Return the fully encrypted onion message

# ONION DECRYPTION IN MESSAGE NODES

In [40]:
def process_onion_message(onion_message, private_keys):
    data=onion_message
    hexadecimal_data=data.hex()
    print()
    print(f"[START NODE] The initial encrypted message is: {hexadecimal_data}")
    print()
    for i in range(len(private_keys)): # Iterate (extracting) over the private keys to decrypt each layer of the onion network
        print(f"[STEP {i+1}] Decrypting layer {i+1}...")
        encrypted_AES_key=data[:256]
        hexadecimal_encrypted_AES_key=encrypted_AES_key.hex()
        print(f"[STEP {i+1}] Layer {i+1} - Extracted AES key (encrypted): {hexadecimal_encrypted_AES_key}")
        nonce=data[256:272]
        hexadecimal_nonce=nonce.hex()
        print(f"[STEP {i+1}] Layer {i+1} - Extracted nonce: {hexadecimal_nonce}")
        cipher_text=data[272:-16]
        hexadecimal_cipher_text=cipher_text.hex()
        print(f"[STEP {i+1}] Layer {i+1} - Extracted text: {hexadecimal_cipher_text}")
        cipher_tag=data[-16:]
        hexadecimal_cipher_tag=cipher_tag.hex()
        print(f"[STEP {i+1}] Layer {i+1} - Extracted tag: {hexadecimal_cipher_tag}")
        # Decrypt the AES key using the current private key
        AES_key=decrypt_with_RSA(encrypted_AES_key, private_keys[i])
        hexadecimal_AES_key=AES_key.hex()
        print(f"[STEP {i+1}] Layer {i+1} - Decrypted AES key: {hexadecimal_AES_key}")
        # Decrypt the message using the decrypted AES key
        data=decrypt_with_AES(nonce, cipher_text, cipher_tag, AES_key)
        hexadecimal_data=data.hex()
        print(f"[STEP {i+1}] Layer {i+1} - Decrypted data (layer {i+1} removed): {hexadecimal_data}")
        print()
    decoded_data=data.decode() # Decode the final decrypted message to string
    print(f"[FINAL NODE] The original message decrypted received is: {decoded_data}")

# FINAL PROGRAM OUTPUT

In [ ]:
def writing_message():
    global message
    message=input("Enter your secret message to send with Tor network (e.g. 'The secret meeting is scheduled for March 25 at 10:00 AM in the underground bunker. The access code is 47A9X.'): ")
    print(f"The secret message that will be sent in the Tor network is the following: {message}")
    confirm_message=input(f"Is the written message '{message}' correct (reply with Y/N only)? ").upper()
    if confirm_message=="Y":
        print("Answer Y has been selected. Then the program will proceed to send the message to the Tor network.")
    elif confirm_message=="N":
        print("Answer N has been selected. So, is possible rewrite the message you want to send in the Tor network.")
        writing_message()
    else:
        print("Invalid response. You can only respond with Y/N. Then the program will ask you to re-enter the message you want to send to the Tor network.")
        writing_message()
writing_message()

The secret message that will be sent in the Tor network is the following: I'm a Python Dev!
Invalid response. You can only respond with Y/N. Then the program will ask you to re-enter the message you want to send to the Tor network.
The secret message that will be sent in the Tor network is the following: I'm a Python Dev!
Answer N has been selected. So, is possible rewrite the message you want to send in the Tor network.
The secret message that will be sent in the Tor network is the following: I'm a Python Dev!
Answer Y has been selected. Then the program will proceed to send the message to the Tor network.


In [ ]:
onion_message=create_onion_message(message, public_RSA_key) # Create the onion message (apply onion routing encryption)
process_onion_message(onion_message, private_RSA_key) # Decrypt the onion message layer by layer using the private key
print()
program_exit=input("Press ENTER to exit the program...")


[START NODE] The initial encrypted message is: a3c43e6411b935809cdb5860e3f85f61d4da691177fd7f9e9d8418f3b2a9b3962761b444a5be5e8cc4f524d2b4471ed8f066132953317484d9c4fa14fed8c2d467c2946a16ba09c18a4ad30a31068ad51f5cf39432d2dd383aeaaf54fd6da913bc8b2d657a8c100e9074fc99f8dcdc210f56bd5530c622b1d9b3a81b039933f7c4b19c3e7b93e1a9e7e42740076d6d74061cc936b0b6ef48664a4ab1b48f3b1608da1fe8c32bb08d4c5f1116303687a70c48fa8add93b2c6fc75c347609e91dd33e28539b1b060ab5bfdcc384a44b7271d96b2337937160a4571e23abc7fbd7a97372dccc191f8b6ce7521dd3cad1ea2dc04a1fb068e6e9f6b1cad861793541205824aabb5268f2bb1b952541eb8b9ee9e4f1572f75dfaca71862338991fda05c7ff5ff39a9ab6c2d53d01e8bc647f5f0ea99bac6c2cedd68261cb98ba23dbe89a6967726f0180826b7eda3631b993e6b2600ebad31cf24dc226710523c22bd16112f88d602157f74e39158e6e03c6959f2e3de3774d94fec51f2f70c55095e6f4730e75d7ecaaae5abf336bae7663f638ed04970e0abd3c30cc1aa57879910b95500d2cae52924db909867c788bdd512de26aa219bf26eb818a6d930572ed375729039eeb9cf8199e781cb08a245bfb148de909ae16a89ae8c3cf32